# Random Forest Cloud — Predicción Local + Kaggle Submission

**Corre localmente** después de descargar el modelo entrenado en Kaggle Notebooks.

**Archivos necesarios en la misma carpeta:**
- `rfr_cloud_best.pkl` — modelo entrenado
- `rfr_cloud_metadata.json` — metadata

In [ ]:
import joblib
import pandas as pd
import numpy as np
import json
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import sklearn

print(f'sklearn version: {sklearn.__version__}')
print('Imports OK')

In [ ]:
MODEL_DIR = Path('.')
DATA_DIR  = Path('../../data/processed')

model_file = MODEL_DIR / 'rfr_cloud_best.pkl'
meta_file  = MODEL_DIR / 'rfr_cloud_metadata.json'

print(f'MODEL_DIR : {MODEL_DIR.resolve()}')
for f in [model_file, meta_file]:
    status = '✓ encontrado' if f.exists() else '✗ NO encontrado — descargar desde Kaggle'
    print(f'  {f.name} : {status}')

In [ ]:
with open(meta_file, 'r') as f:
    meta = json.load(f)

print('Metadata del modelo:')
print(f'  CV AUC (5-fold OOF) : {meta.get("cv_auc", meta.get("best_cv_auc", "N/A"))}')
print(f'  Train AUC (full)    : {meta["train_auc"]}')
print(f'  n_train             : {meta.get("n_train", "N/A"):,}' if meta.get("n_train") else f'  n_train             : N/A')
print(f'  n_features          : {meta.get("n_features", len(meta["feature_cols"]))}')
print(f'  Native NaN          : {meta["native_nan"]}')
print(f'  sklearn version     : {meta["sklearn_version"]}')
print(f'  Timestamp           : {meta["timestamp"]}')
print('  Hiperparámetros:')
for k, v in meta['best_params'].items():
    print(f'    {k:<22}: {v}')

In [ ]:
print(f'Cargando modelo: {model_file.name}  ({model_file.stat().st_size/1e6:.2f} MB)...')
model = joblib.load(model_file)
print('Modelo cargado ✓')
print(f'  n_estimators : {model.n_estimators}')
print(f'  max_depth    : {model.max_depth}')

In [ ]:
print('Cargando features_test.parquet...')
df_test      = pd.read_parquet(DATA_DIR / 'features_test.parquet')
sk_ids_test  = df_test['SK_ID_CURR'].values
feature_cols = meta['feature_cols']

# Encodear categóricas si existen
cat_cols = [c for c in feature_cols if df_test[c].dtype == 'object']
if cat_cols:
    for col in cat_cols:
        cats    = df_test[col].dropna().unique()
        mapping = {v: i for i, v in enumerate(cats)}
        df_test[col] = df_test[col].map(mapping)

X_test = df_test[feature_cols].values

# Imputar NaN si el modelo fue entrenado con sklearn < 1.4
if not meta.get('native_nan', True):
    print('  Imputando NaN (sklearn < 1.4)...')
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy='median')
    X_test = imp.fit_transform(X_test)

print(f'  Test shape  : {X_test.shape}')
print(f'  Features    : {len(feature_cols)}')
print(f'  NaNs en X   : {np.isnan(X_test).sum():,}')

In [ ]:
print('Generando predicciones...')
y_test_prob = model.predict_proba(X_test)[:, 1]

print(f'  Predicciones generadas : {len(y_test_prob):,}')
print(f'  Score mínimo           : {y_test_prob.min():.5f}')
print(f'  Score máximo           : {y_test_prob.max():.5f}')
print(f'  Score medio            : {y_test_prob.mean():.5f}')
print(f'  % predicho como default: {(y_test_prob > 0.5).mean()*100:.2f}%')

In [ ]:
submission = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_test_prob})
out_path   = DATA_DIR / 'submission_cloud_rfr.csv'
submission.to_csv(out_path, index=False)

print(f'Submission guardado: {out_path}')
print(f'Shape: {submission.shape}')
display(submission.head())

## Kaggle Submission — AUC Test (Public Leaderboard)

> **Límite**: 5 submissions/día.

In [ ]:
from kaggle import KaggleApi

COMPETITION = 'home-credit-default-risk'
_cv_auc = meta.get('cv_auc', meta.get('best_cv_auc', 0))
_msg = (f"Random Forest Cloud | "
        f"CV AUC={_cv_auc:.5f} | "
        f"train AUC={meta['train_auc']:.5f} | "
        f"n_est={meta['best_params']['n_estimators']}")

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub    = _as_str(getattr(matched, 'public_score',  None))
            status = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(out_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, out_path.name, _msg)

print(f'\n' + '=' * 65)
print(f'RESULTADO KAGGLE — RANDOM FOREST CLOUD')
print(f'=' * 65)
print(f'  AUC test Public  LB  : {public_auc}')
print(f'  AUC test Private LB  : {private_auc}')
print(f'  CV AUC (entrenamiento): {_cv_auc:.5f}')
if public_auc:
    print(f'  Gap CV - Public LB   : {abs(_cv_auc - public_auc):.5f}')
print(f'=' * 65)